# RAW → BRONZE: Ingestion de Customers

Este notebook ejecuta el script de ingesta que:
1. Se conecta a MySQL via JDBC usando PySpark
2. Lee la tabla `customers`
3. Guarda los datos localmente en formato Parquet (capa Bronze)

In [1]:
import sys
from pathlib import Path

# Agrega el directorio raiz del proyecto al path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/agus/local/multihope


## 1. Verificar configuracion de base de datos

In [2]:
from src.utils.config_loader import load_db_config, get_jdbc_url

db_config = load_db_config()
jdbc_url = get_jdbc_url(db_config)

print(f"Host     : {db_config['host']}")
print(f"Port     : {db_config['port']}")
print(f"Database : {db_config['database']}")
print(f"User     : {db_config['user']}")
print(f"JDBC URL : {jdbc_url}")

Host     : www.bigdataybi.com
Port     : 3306
Database : fake
User     : curso
JDBC URL : jdbc:mysql://www.bigdataybi.com:3306/fake?useSSL=false&allowPublicKeyRetrieval=true


## 2. Ejecutar el script RAW → BRONZE

In [3]:
from src.raw_to_bronze.customers_ingestion import ingest_customers

total_records = ingest_customers()
print(f'\nIngestion completada: {total_records} registros guardados en Bronze.')

2026-04-07 08:42:37,141 [INFO] src.raw_to_bronze.customers_ingestion - === RAW -> BRONZE | customers ===
26/04/07 08:42:38 WARN Utils: Your hostname, Agustins-MacBook-Pro-M4.local resolves to a loopback address: 127.0.0.1; using 192.168.1.3 instead (on interface en0)
26/04/07 08:42:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/agus/.ivy2/cache
The jars for the packages stored in: /Users/agus/.ivy2/jars
mysql#mysql-connector-java added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d8e5222a-b4a6-4b8d-95ee-9bd1615e8cdb;1.0
	confs: [default]
	found mysql#mysql-connector-java;8.0.33 in central
	found com.mysql#mysql-connector-j;8.0.33 in central
	found com.google.protobuf#protobuf-java;3.21.9 in central


:: loading settings :: url = jar:file:/Users/agus/local/multihope/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


:: resolution report :: resolve 71ms :: artifacts dl 2ms
	:: modules in use:
	com.google.protobuf#protobuf-java;3.21.9 from central in [default]
	com.mysql#mysql-connector-j;8.0.33 from central in [default]
	mysql#mysql-connector-java;8.0.33 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   2   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-d8e5222a-b4a6-4b8d-95ee-9bd1615e8cdb
	confs: [default]
	0 artifacts copied, 2 already retrieved (0kB/2ms)
26/04/07 08:42:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Set

AnalysisException: [PATH_ALREADY_EXISTS] Path file:/Users/agus/local/multihope/data/bronze/customers already exists. Set mode as "overwrite" to overwrite the existing path.

26/04/07 08:42:49 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


## 3. Validar la capa Bronze

In [ ]:
from src.utils.spark_session import create_spark_session

BRONZE_PATH = PROJECT_ROOT / 'data' / 'bronze' / 'customers'

spark = create_spark_session('notebook_validation')
df_bronze = spark.read.parquet(str(BRONZE_PATH))

print(f'Schema Bronze:')
df_bronze.printSchema()

Schema Bronze:
root
 |-- id_cliente: integer (nullable = true)
 |-- identificacion: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- email: string (nullable = true)
 |-- telefono: string (nullable = true)
 |-- direccion: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- _loadtime: timestamp (nullable = true)



In [ ]:
print(f'Total registros: {df_bronze.count()}')
df_bronze.show(10, truncate=False)

Total registros: 10
+----------+--------------+-----------------------------+---------------------------+--------------+-----------------------------------------------------------------------------------------------+--------+-------------------+
|id_cliente|identificacion|nombre                       |email                      |telefono      |direccion                                                                                      |estado  |_loadtime          |
+----------+--------------+-----------------------------+---------------------------+--------------+-----------------------------------------------------------------------------------------------+--------+-------------------+
|1         |1705251732    |Josefina Alicia Otero Rosas  |estradaines@icloud.com     |(593)958474054|Prolongación Sudán del Sur 590 Edif. 300 , Depto. 542, San Ricardo de la Montaña, NL 34179-3364|activo  |2025-06-26 19:34:48|
|2         |0700652068    |Dr. Estela Serrano           |ricardopantoja@gmai

In [ ]:
# Estadisticas basicas
df_bronze.describe().show()

26/03/27 09:33:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+-------------------+--------------------+--------------------+------------+--------------------+--------+
|summary|        id_cliente|     identificacion|              nombre|               email|    telefono|           direccion|  estado|
+-------+------------------+-------------------+--------------------+--------------------+------------+--------------------+--------+
|  count|                10|                 10|                  10|                  10|          10|                  10|      10|
|   mean|               5.5|      9.947628328E8|                NULL|                NULL|9.19849754E8|                NULL|    NULL|
| stddev|3.0276503540974917|5.299266023230045E8|                NULL|                NULL|        NULL|                NULL|    NULL|
|    min|                 1|         0101733954|Adalberto María J...|araceli72@hotmail...|            |Ampliación Abrego...|  activo|
|    max|                10|         1709959652| Sr(a). Serafí

In [ ]:
spark.stop()
print('SparkSession cerrada.')

SparkSession cerrada.
